# 5. 운영 판단 기준과 데이터 버전 관리

## 목적

통계적 참고구간과 현장 허용한계를 분리하고, 실시간 입력 가능 변수·종료 후 설명변수·오프라인 변수·결과/라벨을 명시한다. 원본 파일의 해시와 기본 프로파일을 기록하며, 이상치·결측치 변경은 사유와 영향을 남긴 뒤 별도 버전에서만 수행하도록 틀을 만든다. 이 노트북은 파일을 저장하거나 원본을 변경하지 않는다.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


source_path = find_project_root() / 'data/interim/merged_data_ko.csv'
source_hash_before = sha256(source_path)
data = pd.read_csv(source_path)
manifest = pd.DataFrame([{
    '데이터파일': source_path.name, '버전ID_SHA256앞12자리': source_hash_before[:12],
    '파일크기_bytes': source_path.stat().st_size, '행수': len(data), '열수': data.shape[1],
    '배치수': data['배치번호'].nunique(), '완전중복행수': data.duplicated().sum(),
    '총결측셀수': int(data.isna().sum().sum()),
}])
display(manifest)

,데이터파일,버전ID_SHA256앞12자리,파일크기_bytes,행수,열수,배치수,완전중복행수,총결측셀수
0,merged_data_ko.csv,0fab70f7bd22,22351764,113935,39,100,0,559365


### 판단

SHA-256 앞 12자리를 분석 버전 식별자로 사용한다. 같은 파일명이라도 내용이 바뀌면 해시가 달라지므로 분석 재현 시 파일명과 해시를 함께 기록해야 한다.

In [2]:
identifier_columns = ['배치번호', '배치참조번호']
time_columns = ['발효시간(h)']
offline_columns = [
    '페니실린농도_오프라인(g/L)', '바이오매스농도_오프라인(g/L)', '점도_오프라인(cP)',
    'PAA농도_오프라인(g/L)', '암모니아농도_오프라인(g/L)',
]
result_or_label_columns = [
    '중간수확량(kg)', '종료수확량(kg)', '총수확량(kg)',
    '구간결함(0:정상,1:결함)', '배치결함(0:정상,1:결함)',
]
online_columns = [column for column in data.columns
                  if column not in identifier_columns + time_columns + offline_columns + result_or_label_columns]

registry_rows = []
for order, column in enumerate(data.columns, start=1):
    if column in identifier_columns:
        category, realtime_use, model_rule = '식별자', False, '모델 입력 금지; 분할·추적에만 사용'
    elif column in time_columns:
        category, realtime_use, model_rule = '현재시간', True, '현재까지 경과시간으로 사용 가능'
    elif column in offline_columns:
        category, realtime_use, model_rule = '오프라인', False, '실시간 모델 입력 금지'
    elif column in result_or_label_columns:
        category, realtime_use, model_rule = '결과/라벨', False, '예측대상 또는 평가에만 사용'
    else:
        category, realtime_use, model_rule = '온라인 공정변수', True, '시점 t까지 실제 관측 가능할 때만 사용'
    registry_rows.append({
        '순서': order, '컬럼': column, '구분': category, '실시간사용가능': realtime_use,
        '모델사용규칙': model_rule, '결측률(%)': data[column].isna().mean() * 100,
    })
column_registry = pd.DataFrame(registry_rows)
display(column_registry.groupby(['구분', '실시간사용가능']).size().rename('컬럼수').to_frame())
display(column_registry.loc[column_registry['컬럼'].eq('오일유량(L/h)')])

,,컬럼수
구분,실시간사용가능,
결과/라벨,False,5
식별자,False,2
오프라인,False,5
온라인 공정변수,True,26
현재시간,True,1


,순서,컬럼,구분,실시간사용가능,모델사용규칙,결측률(%)
24,25,오일유량(L/h),온라인 공정변수,True,시점 t까지 실제 관측 가능할 때만 사용,0.0


### 판단

오일유량은 원본 정의상 온라인 공정변수이므로 시점 t에 실제 계측·기록된다면 시간적 누수는 아니다. 하지만 배치 전체 누적값이나 종료 후 보정값을 입력하면 누수가 된다. 또한 오일유량이 전략·레시피·결과를 거의 완벽히 대리하는지는 시간기반 교차검증으로 별도 확인해야 한다. 오프라인 5개와 결과/결함 5개는 실시간 입력에서 제외한다.

In [3]:
derived_registry = pd.DataFrame([
    {'파생변수': '시점t까지_산누적량', '계산범위': '0~현재시점 t', '실시간가능': True, '주의': '배치 전체 산총투입량을 사용하면 누수'},
    {'파생변수': '최근20시간_기울기', '계산범위': 't-20~t', '실시간가능': True, '주의': '창 끝이 현재시점을 넘지 않아야 함'},
    {'파생변수': '후기기질기울기', '계산범위': '실제 종료시간의 마지막 20%', '실시간가능': False, '주의': '종료시간을 미리 알아야 하므로 설명용'},
    {'파생변수': '농도유지율', '계산범위': '최종농도/배치최대농도', '실시간가능': False, '주의': '배치 종료 후 결과지표'},
    {'파생변수': '종료시간', '계산범위': '배치 최종 관측시간', '실시간가능': False, '주의': '미래정보'},
])
display(derived_registry)

decision_levels = pd.DataFrame([
    {'단계': '통계적 참고밴드', '근거': '중앙값·IQR·분위수', '용도': '패턴 비교와 조사 신호', '자동조치': False},
    {'단계': '경보한계', '근거': '독립 검증에서 민감도·오경보율 합의', '용도': '운영자 확인 요청', '자동조치': False},
    {'단계': '조치한계', '근거': '공정·장비·품질 허용범위와 승인', '용도': 'SOP에 따른 조치', '자동조치': '승인된 SOP에 한함'},
])
display(decision_levels)

,파생변수,계산범위,실시간가능,주의
0,시점t까지_산누적량,0~현재시점 t,True,배치 전체 산총투입량을 사용하면 누수
1,최근20시간_기울기,t-20~t,True,창 끝이 현재시점을 넘지 않아야 함
2,후기기질기울기,실제 종료시간의 마지막 20%,False,종료시간을 미리 알아야 하므로 설명용
3,농도유지율,최종농도/배치최대농도,False,배치 종료 후 결과지표
4,종료시간,배치 최종 관측시간,False,미래정보


,단계,근거,용도,자동조치
0,통계적 참고밴드,중앙값·IQR·분위수,패턴 비교와 조사 신호,False
1,경보한계,독립 검증에서 민감도·오경보율 합의,운영자 확인 요청,False
2,조치한계,공정·장비·품질 허용범위와 승인,SOP에 따른 조치,승인된 SOP에 한함


### 판단

통계적으로 유의하거나 골든배치 IQR 밖이라는 이유만으로 공정을 중단하면 안 된다. 통계밴드는 조사 신호, 경보한계는 운영자 확인, 조치한계는 승인된 SOP 실행으로 역할을 분리한다.

In [4]:
change_log_template = pd.DataFrame(columns=[
    '변경ID', '데이터버전_이전', '데이터버전_이후', '변경일시', '담당자',
    '대상컬럼', '대상배치', '대상시간', '원본값', '변경값', '변경유형',
    '변경근거', '영향행수', '분석영향', '승인상태', '복구방법',
])
display(change_log_template)

source_hash_after = sha256(source_path)
assert source_hash_before == source_hash_after, '분석 중 원본 파일이 변경되었습니다.'
print('원본 해시 유지 확인:', source_hash_after[:12])
print('이 노트북에서는 데이터 또는 로그 파일을 저장하지 않았습니다.')

,변경ID,데이터버전_이전,데이터버전_이후,변경일시,담당자,대상컬럼,대상배치,대상시간,원본값,변경값,변경유형,변경근거,영향행수,분석영향,승인상태,복구방법


원본 해시 유지 확인: 0fab70f7bd22
이 노트북에서는 데이터 또는 로그 파일을 저장하지 않았습니다.


### 최종 판단과 결론

- 현재 데이터는 해시·크기·행·열·배치수로 식별할 수 있어 분석 버전 추적이 가능하다. 노트북 실행 전후 해시가 동일해 원본은 변경되지 않았다.
- 실시간 모델은 온라인 공정변수와 현재시점까지 계산한 파생변수만 사용해야 한다. 오프라인 변수, 결과/결함 라벨, 실제 종료시간 기반 파생변수는 입력에서 제외한다.
- 이상치·결측치를 변경하려면 원본을 덮어쓰지 말고 변경 사유·범위·영향·승인·복구방법을 로그에 남긴 새 버전을 만들어야 한다. 현재는 사용자 지시에 따라 로그 틀만 메모리에 만들고 저장하지 않았다.
- 통계적 참고밴드는 마련됐지만 현장 경보·조치한계는 아직 확정되지 않았다. 도메인 허용범위와 독립 검증 성능이 승인되기 전까지 현재 기준은 분석 참고용이다.